# Nano Banana - Image Generation with Google Gemini

This notebook demonstrates how to generate and edit images using **Nano Banana**, the name for Gemini's native image generation capabilities. We'll cover text-to-image generation, image editing, style transfer, multi-turn conversations, and advanced features like Google Search grounding.

## What is Nano Banana?

Nano Banana is the name for Google's native image generation capabilities built into the Gemini model family. Gemini can generate and process images conversationally with text, images, or a combination of both.

There are two models available:

| Model | Model ID | Best For |
|-------|----------|----------|
| **Nano Banana** | `gemini-2.5-flash-image` | Speed and efficiency. Optimized for high-volume, low-latency tasks. Generates at 1024px. Works best with up to 3 input images. |
| **Nano Banana Pro** | `gemini-3-pro-image-preview` | Professional asset production. Uses advanced reasoning ("Thinking") to follow complex instructions. Supports up to 4K resolution and 14 reference images. |

Key capabilities:
- **Text-to-image generation** from natural language prompts
- **Image editing** - add/remove elements, inpainting, style transfer
- **Multi-image composition** combining multiple reference images
- **Accurate text rendering** in generated images (especially Pro)
- **Configurable output** with aspect ratios and resolutions up to 4K
- **Grounding with Google Search** for real-world context (Pro only)
- All generated images include a **SynthID watermark**

## Setup

In [ ]:
%pip install -q google-genai Pillow python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY must be set in your .env file"

In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from IPython.display import display

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

We'll use this helper throughout the notebook to display and optionally save images from the model response.

In [ ]:
def display_response(response, save_as=None):
    """Display text and images from a Gemini response."""
    images = []
    for part in response.parts:
        if getattr(part, "thought", False):
            # Skip thinking parts (Nano Banana Pro)
            continue
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            image = part.as_image()
            display(image)
            images.append(image)
            if save_as:
                image.save(save_as)
    return images

---

## 1. Text-to-Image Generation

The simplest use case: describe what you want and the model generates it. The fundamental principle for good results is to **describe the scene, don't just list keywords**. A narrative, descriptive paragraph will almost always produce a better image than disconnected words.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A cozy coffee shop on a rainy afternoon, watercolor style",
)

display_response(response, save_as="coffee_shop.png")

### Photorealistic Images

For realistic images, use photography terminology: camera angles, lens types, lighting, and fine details.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A photorealistic close-up portrait of an elderly Japanese ceramicist "
        "with deep, sun-etched wrinkles and a warm, knowing smile. He is carefully "
        "inspecting a freshly glazed tea bowl. The setting is his rustic, sun-drenched "
        "workshop with pottery wheels and shelves of clay pots in the background. "
        "Illuminated by soft, golden hour light streaming through a window. "
        "Captured with an 85mm portrait lens, soft blurred background (bokeh)."
    ),
)

display_response(response, save_as="ceramicist_portrait.png")

### Stylized Illustrations and Stickers

To create stickers, icons, or assets, be explicit about the style and request a white/transparent background.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A kawaii-style sticker of a happy red panda wearing a tiny bamboo hat. "
        "It's munching on a green bamboo leaf. The design features bold, clean outlines, "
        "simple cel-shading, and a vibrant color palette. The background must be white."
    ),
)

display_response(response, save_as="red_panda_sticker.png")

### Product Mockups

Create clean, professional product shots for e-commerce or advertising.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A high-resolution, studio-lit product photograph of a minimalist ceramic "
        "coffee mug in matte black, presented on a polished concrete surface. "
        "The lighting is a three-point softbox setup designed to create soft, diffused "
        "highlights and eliminate harsh shadows. The camera angle is a slightly elevated "
        "45-degree shot to showcase its clean lines. Ultra-realistic, with sharp focus "
        "on the steam rising from the coffee. Square image."
    ),
)

display_response(response, save_as="product_mockup.png")

---

## 2. Configuring Aspect Ratio and Resolution

You can control the output dimensions using `ImageConfig`.

**Available aspect ratios:** `1:1`, `2:3`, `3:2`, `3:4`, `4:3`, `4:5`, `5:4`, `9:16`, `16:9`, `21:9`

**Available resolutions (Pro only):** `1K`, `2K`, `4K`

| Model | Default Resolution | Max Resolution |
|-------|-------------------|----------------|
| `gemini-2.5-flash-image` | 1024px (fixed) | 1024px |
| `gemini-3-pro-image-preview` | 1K | 4K |

> **Note:** You must use an uppercase `K` (e.g., `1K`, `2K`, `4K`). Lowercase will be rejected.

In [ ]:
# Wide landscape with 16:9 aspect ratio
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A panoramic mountain landscape at sunset with dramatic clouds and a winding river",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    ),
)

display_response(response, save_as="mountain_landscape.png")

In [ ]:
# Tall portrait with 9:16 aspect ratio and 2K resolution (Pro)
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=(
        "A minimalist composition featuring a single, delicate red maple leaf "
        "positioned in the bottom-right of the frame. The background is a vast, "
        "empty off-white canvas, creating significant negative space for text. "
        "Soft, diffused lighting from the top left."
    ),
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="9:16",
            image_size="2K",
        ),
    ),
)

display_response(response, save_as="minimalist_leaf.png")

---

## 3. Image Editing

Provide an existing image alongside a text prompt to edit it. The model can add/remove elements, change styles, adjust color grading, and more. We'll generate a base image first, then apply several different edits to it.

In [ ]:
# Generate a base image to use for editing
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A photorealistic picture of a fluffy ginger cat sitting on a wooden floor, "
        "looking directly at the camera. Soft, natural light from a window."
    ),
)

print("Base image:")
base_images = display_response(response, save_as="base_cat.png")
base_image = base_images[0] if base_images else None

### Adding Elements

Describe what you want to add. The model will match the original image's style, lighting, and perspective.

In [ ]:
if base_image:
    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[
            "Using the provided image of my cat, please add a small, knitted wizard hat "
            "on its head. Make it look like it's sitting comfortably and not falling off.",
            base_image,
        ],
    )

    print("With wizard hat:")
    display_response(response, save_as="cat_wizard_hat.png")

### Inpainting (Semantic Masking)

You can conversationally define a "mask" to edit a specific part of an image while leaving the rest untouched. No need for an actual mask image - just describe what to change.

In [ ]:
# Generate a living room for inpainting
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A wide shot of a modern, well-lit living room with a prominent blue sofa "
        "in the center. A coffee table is in front of it and a large window is in the background."
    ),
)

print("Original living room:")
room_images = display_response(response, save_as="living_room.png")
room_image = room_images[0] if room_images else None

In [ ]:
# Inpaint: change only the sofa
if room_image:
    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[
            room_image,
            "Using the provided image of a living room, change only the blue sofa to be "
            "a vintage, brown leather chesterfield sofa. Keep the rest of the room, "
            "including the pillows on the sofa and the lighting, unchanged.",
        ],
    )

    print("After inpainting (sofa replaced):")
    display_response(response, save_as="living_room_inpainted.png")

### Style Transfer

Provide an image and ask the model to recreate its content in a different artistic style.

In [ ]:
# Generate a city scene and apply style transfer
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=(
        "A photorealistic, high-resolution photograph of a busy city street "
        "in New York at night, with bright neon signs, yellow taxis, and tall skyscrapers."
    ),
)

print("Original city photo:")
city_images = display_response(response, save_as="city_original.png")
city_image = city_images[0] if city_images else None

In [ ]:
if city_image:
    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[
            city_image,
            "Transform the provided photograph of a modern city street at night into "
            "the artistic style of Vincent van Gogh's 'Starry Night'. Preserve the original "
            "composition of buildings and cars, but render all elements with swirling, "
            "impasto brushstrokes and a dramatic palette of deep blues and bright yellows.",
        ],
    )

    print("Van Gogh style transfer:")
    display_response(response, save_as="city_van_gogh.png")

---

## 4. Accurate Text Rendering (Nano Banana Pro)

Nano Banana Pro excels at rendering legible, stylized text for infographics, logos, menus, diagrams, and marketing assets. For best results, be clear about the exact text, font style, and overall design.

In [ ]:
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=(
        "Create a modern, minimalist logo for a coffee shop called 'The Daily Grind'. "
        "The text should be in a clean, bold, sans-serif font. The color scheme is "
        "black and white. Put the logo in a circle. Use a coffee bean in a clever way."
    ),
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
        ),
    ),
)

display_response(response, save_as="coffee_logo.png")

In [ ]:
# Da Vinci style sketch with text annotations
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=(
        "Da Vinci style anatomical sketch of a dissected Monarch butterfly. "
        "Detailed drawings of the head, wings, and legs on textured parchment "
        "with notes in English."
    ),
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
            image_size="2K",
        ),
    ),
)

display_response(response, save_as="butterfly_sketch.png")

---

## 5. Multi-Turn Conversation

Using the chat interface, you can iteratively refine images across multiple turns. This is the recommended way to iterate on images. The chat handles thought signatures automatically, preserving reasoning context between turns.

In [ ]:
chat = client.chats.create(
    model="gemini-3-pro-image-preview",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

# Turn 1: Generate an infographic
response = chat.send_message(
    "Create a vibrant infographic that explains photosynthesis as if it were a recipe "
    "for a plant's favorite food. Show the 'ingredients' (sunlight, water, CO2) and "
    "the 'finished dish' (sugar/energy). The style should be like a page from a colorful "
    "kids' cookbook, suitable for a 4th grader."
)

print("Turn 1 - English version:")
display_response(response, save_as="photosynthesis_en.png")

In [ ]:
# Turn 2: Translate the infographic to Spanish
response = chat.send_message(
    "Update this infographic to be in Spanish. Do not change any other elements of the image.",
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="2K",
        ),
    ),
)

print("Turn 2 - Spanish version:")
display_response(response, save_as="photosynthesis_es.png")

In [ ]:
# Turn 3: Further refinement
response = chat.send_message(
    "Make the background darker and add a border around the whole image."
)

print("Turn 3 - Refined:")
display_response(response, save_as="photosynthesis_refined.png")

---

## 6. Multi-Image Composition

Nano Banana Pro supports up to **14 reference images** (6 objects + 5 humans for consistency), enabling advanced composition and subject consistency.

In [ ]:
# Generate two reference images
ref_images = []
ref_prompts = [
    "A professionally shot photo of a blue floral summer dress on a plain white background, ghost mannequin style.",
    "Full-body shot of a woman with her hair in a bun, smiling, standing against a neutral grey studio background.",
]

for i, prompt in enumerate(ref_prompts):
    resp = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=prompt,
    )
    for part in resp.parts:
        if part.inline_data is not None:
            img = part.as_image()
            ref_images.append(img)
            print(f"Reference image {i + 1}:")
            display(img)

print(f"\nGenerated {len(ref_images)} reference images")

In [ ]:
# Compose them together
if len(ref_images) == 2:
    response = client.models.generate_content(
        model="gemini-3-pro-image-preview",
        contents=[
            "Create a professional e-commerce fashion photo. Take the blue floral dress "
            "from the first image and let the woman from the second image wear it. "
            "Generate a realistic, full-body shot of the woman wearing the dress, with "
            "the lighting and shadows adjusted to match an outdoor garden environment.",
            ref_images[0],
            ref_images[1],
        ],
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"],
            image_config=types.ImageConfig(
                aspect_ratio="2:3",
                image_size="2K",
            ),
        ),
    )

    print("Composed result:")
    display_response(response, save_as="fashion_composite.png")

---

## 7. Grounding with Google Search (Nano Banana Pro)

Nano Banana Pro can use Google Search to ground its image generation in **real-time data**. This is useful for weather forecasts, recent events, sports scores, or any time-sensitive information.

The response includes `groundingMetadata` with search suggestions and the top 3 web sources used.

In [ ]:
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=(
        "Visualize the current weather forecast for the next 5 days in San Francisco "
        "as a clean, modern weather chart. Add a visual on what I should wear each day."
    ),
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(aspect_ratio="16:9"),
        tools=[{"google_search": {}}],
    ),
)

display_response(response, save_as="weather_forecast.png")

---

## 8. Bringing Sketches to Life (Nano Banana Pro)

You can upload a rough sketch or drawing and ask the model to refine it into a polished, finished image. We'll generate a sketch first to simulate this workflow.

In [ ]:
# Generate a rough sketch
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A rough pencil sketch of a flat sports car on white paper, simple lines only",
)

print("Sketch:")
sketch_images = display_response(response, save_as="car_sketch.png")
sketch = sketch_images[0] if sketch_images else None

In [ ]:
# Turn the sketch into a polished render
if sketch:
    response = client.models.generate_content(
        model="gemini-3-pro-image-preview",
        contents=[
            sketch,
            "Turn this rough pencil sketch of a futuristic car into a polished photo "
            "of the finished concept car in a showroom. Keep the sleek lines and low "
            "profile from the sketch but add metallic blue paint and neon rim lighting.",
        ],
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"],
            image_config=types.ImageConfig(
                aspect_ratio="16:9",
                image_size="2K",
            ),
        ),
    )

    print("Polished render:")
    display_response(response, save_as="car_render.png")

---

## 9. Prompting Best Practices

Here are strategies from the official documentation to get the best results:

1. **Be hyper-specific** - Instead of "fantasy armor", describe it: "ornate elven plate armor, etched with silver leaf patterns, with a high collar and pauldrons shaped like falcon wings."

2. **Provide context and intent** - Explain the purpose of the image. "Create a logo for a high-end, minimalist skincare brand" yields better results than just "create a logo."

3. **Iterate and refine** - Use multi-turn conversations to make small changes: "That's great, but can you make the lighting a bit warmer?"

4. **Use step-by-step instructions** - For complex scenes: "First, create a background of a serene, misty forest at dawn. Then, in the foreground, add a moss-covered ancient stone altar. Finally, place a single, glowing sword on top of the altar."

5. **Use semantic negative prompts** - Instead of "no cars", describe the desired scene positively: "an empty, deserted street with no signs of traffic."

6. **Control the camera** - Use photographic and cinematic language: wide-angle shot, macro shot, low-angle perspective, 85mm lens, etc.

In [ ]:
# Demonstrating a well-crafted prompt using these best practices
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=(
        # Step-by-step + hyper-specific + camera control
        "Present a clear, 45-degree top-down isometric miniature 3D cartoon scene of London, "
        "featuring its most iconic landmarks and architectural elements. Use soft, refined "
        "textures with realistic PBR materials and gentle, lifelike lighting and shadows. "
        "Use a clean, minimalistic composition with a soft, solid-colored background. "
        "At the top-center, place the title 'London' in large bold text, "
        "then the date (small text) and temperature (medium text). "
        "All text must be centered with consistent spacing."
    ),
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="9:16",
            image_size="2K",
        ),
    ),
)

display_response(response, save_as="london_isometric.png")

---

## 10. Model Selection Guide

| Feature | Nano Banana (`gemini-2.5-flash-image`) | Nano Banana Pro (`gemini-3-pro-image-preview`) |
|---------|---------------------------------------|-----------------------------------------------|
| Speed | Fast | Slower (uses Thinking) |
| Max resolution | 1024px | Up to 4K |
| Input images | Up to 3 (best results) | Up to 14 (5 humans + 6 objects) |
| Text rendering | Basic | Advanced (legible, stylized) |
| Google Search grounding | No | Yes |
| Thinking / reasoning | No | Yes (enabled by default) |
| Best for | Quick iterations, stickers, icons, simple edits | Professional assets, infographics, logos, complex composition |

### Limitations

- Best language support: EN, ar-EG, de-DE, es-MX, fr-FR, hi-IN, id-ID, it-IT, ja-JP, ko-KR, pt-BR, ru-RU, ua-UA, vi-VN, zh-CN
- No audio or video input support
- The model may not always follow the exact number of requested output images
- All generated images include a SynthID watermark

## Summary

In this notebook we covered:

- **Text-to-image generation** with different styles: photorealistic, illustrations, product mockups
- **Aspect ratio and resolution configuration** for precise output control
- **Image editing** techniques: adding elements, inpainting, and style transfer
- **Accurate text rendering** using Nano Banana Pro for logos and annotated images
- **Multi-turn conversations** for iterative image refinement
- **Multi-image composition** for combining reference images into new scenes
- **Google Search grounding** for generating images based on real-world data
- **Sketch-to-render** for turning rough drawings into polished images
- **Prompting best practices** for getting the best results